# Day 6 Demo 1 - From an LLM call to a multi-agent system, **without a framework**

Everything here is raw **Amazon Bedrock Converse**, so you can see every moving part. The ladder, each rung building on the last:

1. an LLM call -> 2. an LLM workflow -> 3. an agent -> 4. an agent with tool calls -> 5. multi-agent -> 6. multi-agent with tool calls

The same ladder is rebuilt with **Strands** in Demo 2, and **hosted on AgentCore** in Demo 3. This notebook proves the concepts work on real Bedrock by hand.

## Before you run (console + setup, in this order)

1. **Enable model access** - Bedrock console -> **Model catalog** -> enable **Anthropic Claude Sonnet 4.5** in region **us-west-2**. (The old "Model access" page is retired.)
2. **Credentials** - `aws configure` (or environment variables / an attached role). Verify with `aws sts get-caller-identity`.
3. **Set the placeholders in the next cell** - `REGION` and `MODEL`. `MODEL` must be a **`us.` inference profile** id; a bare model id raises `ValidationException`.

> `OFFLINE_DEMO = True` runs the *orchestration* against a local stub (no AWS) so you can dry-run the logic. For a real proof-of-concept, keep it **False** (the default).

In [ ]:
# ---- configuration: EDIT THESE ----
OFFLINE_DEMO = False   # False = call real Bedrock (default). True = offline stub for dry-runs only.
REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"   # <-- a us. inference profile you have ENABLED

import json

In [ ]:
# ---- Bedrock client (lazy) + an offline stub so the notebook is testable without AWS ----
class _Stub:
    """Offline stand-in for bedrock-runtime.converse (dry-runs only; not a real model)."""
    def converse(self, modelId=None, messages=None, toolConfig=None, system=None, inferenceConfig=None, **kw):
        if toolConfig:
            done = sum(1 for m in messages for c in m.get("content", [])
                       if isinstance(c, dict) and "toolResult" in c)
            plan = [("lookup_booking", {"pnr": "JX48Q2"}, "tu_1"),
                    ("get_disruption_reason", {"pnr": "JX48Q2"}, "tu_2"),
                    ("get_rebooking_options", {"pnr": "JX48Q2"}, "tu_3")]
            names = [t["toolSpec"]["name"] for t in toolConfig["tools"]]
            plan = [p for p in plan if p[0] in names]
            if done < len(plan):
                n, i, tid = plan[done]
                return {"output": {"message": {"role": "assistant",
                            "content": [{"toolUse": {"toolUseId": tid, "name": n, "input": i}}]}},
                        "stopReason": "tool_use", "usage": {}}
        last = messages[-1]["content"][-1].get("text", "") if messages else ""
        return {"output": {"message": {"role": "assistant",
                    "content": [{"text": "[offline stub] " + last[:80]}]}},
                "stopReason": "end_turn", "usage": {}}

_client = None
def get_client():
    global _client
    if OFFLINE_DEMO:
        return _Stub()
    if _client is None:
        import boto3
        _client = boto3.client("bedrock-runtime", region_name=REGION)
    return _client

print("mode:", "OFFLINE stub" if OFFLINE_DEMO else ("REAL Bedrock - " + REGION))

## 1 - A single LLM call

One request, one reply. No memory, no tools, no loop - a function call to a model.

In [ ]:
def converse(messages, system=None, tools=None, temperature=0.2, max_tokens=512):
    kw = {"modelId": MODEL, "messages": messages,
          "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature}}
    if system: kw["system"] = [{"text": system}]
    if tools:  kw["toolConfig"] = {"tools": tools}
    return get_client().converse(**kw)

def ask(prompt, system=None, **kw):
    r = converse([{"role": "user", "content": [{"text": prompt}]}], system=system, **kw)
    return r["output"]["message"]["content"][0]["text"]

print(ask("In one sentence, what is an inference profile in Amazon Bedrock?"))

**Best practices**
- Put behaviour/policy in `system`, the task in the user turn - never mix them.
- `temperature=0` for extraction/classification; raise it only for creative drafting.
- Always use the `us.` inference-profile id for newer Claude models.
- Output tokens cost ~5x input - cap `maxTokens`.

**Alternatives**
- `invoke_model` (older, model-specific JSON). Prefer `converse` for a uniform API and tool use.
- Add `guardrailConfig` to enforce a content policy on every call.

## 2 - An LLM *workflow* (a fixed pipeline)

Several calls chained in an order **you** decide. The model does not choose the path. Deterministic and cheap.
Example: a customer message -> (a) extract structured fields -> (b) draft a reply from those fields.

In [ ]:
def workflow(customer_msg):
    extracted = ask('Extract the PNR and intent. Reply with ONLY JSON like {"pnr": "...", "intent": "..."}. Message: ' + customer_msg,
                    temperature=0)
    draft = ask("Write a warm 2-sentence reply to the customer using these details: " + extracted,
                temperature=0.3)
    return extracted, draft

ex, reply = workflow("Hi, my flight on PNR JX48Q2 got cancelled. What now?")
print("STEP 1 extracted:", ex)
print("STEP 2 reply    :", reply)

**Best practices**
- Make early steps deterministic (`temperature=0`) and **validate** their output (e.g. `json.loads`) before the next step.
- Keep each step single-purpose - easier to test than one mega-prompt.
- A workflow fits when the steps are known in advance.

**Alternatives**
- One prompt that does everything: simpler, but brittle and hard to validate mid-way.
- Graduate to an *agent* (next) when the path depends on what the model finds.

## 3 - An *agent* (a model-driven loop), still no tools

The difference from a workflow: the **model** decides when the work is done. Here a reflection loop drafts, critiques, and revises until the model signals `DONE` (or a guard stops it).

In [ ]:
def reflect_agent(task, max_rounds=3):
    draft = ask("Draft a response to this task: " + task, temperature=0.5)
    for rnd in range(max_rounds):
        verdict = ask("You are a strict reviewer. If the draft fully satisfies the task, reply EXACTLY 'DONE'. "
                      "Otherwise reply with ONE concrete improvement. TASK: " + task + " | DRAFT: " + draft,
                      temperature=0)
        if verdict.strip().upper().startswith("DONE"):
            print("  round", rnd + 1, "-> DONE")
            break
        print("  round", rnd + 1, "-> revising:", verdict[:70])
        draft = ask("Improve the draft using this feedback: " + verdict + " | DRAFT: " + draft, temperature=0.4)
    return draft

print(reflect_agent("Explain to a traveler, in 3 short bullets, what to do when a flight is cancelled due to weather."))

**Best practices**
- Always cap the loop (`max_rounds`) - a model can otherwise loop forever.
- Use an explicit, parseable stop signal (`DONE`) and a deterministic reviewer (`temperature=0`).
- This *is* agentic: control flow depends on the model's own output.

**Alternatives**
- Other no-tool agent patterns: plan-then-execute, task decomposition.
- The common case gives the model **tools** (next) so it can act, not just reflect.

## 4 - An agent **with tool calls** (the tool-use loop)

Now the model can *act*: it asks for a tool, you run it, you return the result, and the loop continues until `end_turn`. This is the Day-6 hand-rolled loop.

In [ ]:
# tools (mock data; the model still drives the conversation)
def lookup_booking(pnr):        return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}
def get_disruption_reason(pnr): return {"pnr": pnr, "reason": "weather", "detail": "Heavy fog at origin"}
def get_rebooking_options(pnr): return {"pnr": pnr, "options": [{"flight": "AI-318", "dep": "18:40"},
                                                                {"flight": "6E-552", "dep": "21:15"}]}
TOOLS = {"lookup_booking": lookup_booking, "get_disruption_reason": get_disruption_reason,
         "get_rebooking_options": get_rebooking_options}

def _spec(name, desc):
    return {"toolSpec": {"name": name, "description": desc,
            "inputSchema": {"json": {"type": "object",
                "properties": {"pnr": {"type": "string", "description": "6-char PNR"}}, "required": ["pnr"]}}}}
TOOL_SPECS = [_spec("lookup_booking", "Look up a booking by its PNR."),
              _spec("get_disruption_reason", "Why a booking was delayed or cancelled."),
              _spec("get_rebooking_options", "Alternative flights for a disrupted booking.")]
SYSTEM = "You are TravelMind, a booking-exception assistant. Never invent a PNR."

def tool_agent(user_text, max_steps=6):
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    resp = converse(messages, system=SYSTEM, tools=TOOL_SPECS)
    steps = 0
    while resp["stopReason"] == "tool_use" and steps < max_steps:
        steps += 1
        tu = resp["output"]["message"]["content"][-1]["toolUse"]
        messages.append(resp["output"]["message"])                      # assistant tool_use turn FIRST
        try:
            out = TOOLS[tu["name"]](**tu["input"])
        except Exception as e:
            out = {"error": str(e)}                                     # return the error AS a toolResult
        print("  step", steps, "->", tu["name"], tu["input"], "->", out)
        messages.append({"role": "user", "content": [{"toolResult": {
            "toolUseId": tu["toolUseId"], "content": [{"json": out}]}}]})
        resp = converse(messages, system=SYSTEM, tools=TOOL_SPECS)
    return resp["output"]["message"]["content"][0]["text"]

print(tool_agent("My flight on JX48Q2 was disrupted - what happened and what are my options?"))

**Best practices**
- **Append the assistant `tool_use` message before** the `toolResult` - order is user -> assistant(tool_use) -> user(toolResult).
- **Echo the exact `toolUseId`**; mismatches are rejected.
- **Guard the loop** (`max_steps`) and return tool errors *as* a `toolResult` so the agent recovers.

**Alternatives**
- A Knowledge-Base tool uses the **`bedrock-agent-runtime`** client (`retrieve`), not `bedrock-runtime`.
- Strands writes this entire loop for you (Demo 2).

## 5 - *Multi-agent*, no tools (an orchestrator + specialists)

Several single-purpose agents, coordinated by your code. A triage step routes to a specialist; a final step polishes. Each agent has its own role (system prompt).

In [ ]:
def specialist(role, user, temperature=0.3):
    return ask(user, system=role, temperature=temperature)

def multi_agent(query):
    category = ask("Classify into exactly one word: billing, disruption, or general. Query: " + query,
                   system="You are a triage router. Reply with one word only.", temperature=0)
    cat = category.strip().lower()
    print("  triage ->", cat)
    if "disruption" in cat:
        ans = specialist("You are a flight-disruption specialist. Be specific and practical.", query)
    elif "billing" in cat:
        ans = specialist("You are a billing specialist. Be precise about refunds and fees.", query)
    else:
        ans = specialist("You are a general support agent.", query)
    polished = specialist("You rewrite replies to be warm, clear, and under 60 words.", ans)
    return cat, polished

cat, reply = multi_agent("My flight was cancelled and I want to know my options.")
print("FINAL:", reply)

**Best practices**
- Give each agent a **narrow** role; the orchestrator owns the routing logic.
- More agents = more calls = more cost/latency - split only when roles genuinely differ.
- Keep the routing step deterministic (`temperature=0`).

**Alternatives**
- A single agent with one rich prompt (cheaper, less modular).
- In Strands, specialists become **tools** an orchestrator calls, or a **Swarm/Graph** (Demo 2).

## 6 - Multi-agent **with tool calls**

Combine 4 and 5: the orchestrator routes, and the specialist that must *act* (disruption) is the tool-using loop from stage 4.

In [ ]:
def multi_agent_with_tools(query):
    category = ask("Classify into exactly one word: billing, disruption, or general. Query: " + query,
                   system="You are a triage router. Reply with one word only.", temperature=0)
    cat = category.strip().lower()
    print("  triage ->", cat)
    if "disruption" in cat:
        ans = tool_agent(query)                       # the tool-using agent (stage 4)
    elif "billing" in cat:
        ans = specialist("You are a billing specialist.", query)
    else:
        ans = specialist("You are a general support agent.", query)
    return specialist("You rewrite replies to be warm, clear, and under 60 words.", ans)

print(multi_agent_with_tools("Flight on JX48Q2 cancelled - what happened and what can I do?"))

**Best practices**
- Give tools **only** to agents that need them; keep the others tool-free.
- Cost compounds: a tool loop inside a multi-agent system multiplies model calls - log steps.
- Bound every loop and every handoff.

**Alternatives**
- A single agent with all tools + a strong system prompt (simpler, less separation of concerns).
- Strands `Swarm` / `GraphBuilder` for declarative multi-agent coordination (Demo 2).

## Recap

You built the whole ladder by hand on raw Bedrock: **call -> workflow -> agent -> agent+tools -> multi-agent -> multi-agent+tools**. The jump from a *workflow* (you decide the path) to an *agent* (the model decides) is the key idea. Demo 2 rebuilds this with **Strands**; Demo 3 **hosts** it on AgentCore.